# Repro notebook spec: agm-analysis.ipynb


### 0) Environment
Packages: pandas, numpy, scipy, statsmodels, openpyxl


In [12]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
pd.set_option("display.max_columns", 100)


### 1) Load data
File: agm.xlsx
Sheet: Sheet1


In [13]:
df = pd.read_excel("../data/agm.xlsx", sheet_name="Sheet1")
print(df.shape)
print(df.columns.tolist())


(201, 19)
['age', 'sex', 'UnencryptedGender', 'HighestEducation', 'MemoryProblems', 'APOE', 'CANTABDate', 'PALTEA28z', 'date', 'dm', 'slb', 'awm_freq', 'awm_am', 'gmsi_tot', 'gmsi_ae', 'gmsi_sa', 'gmsi_pa', 'gmsi_mt', 'gmsi_e']


### 2) Variable mapping & preprocessing


In [14]:
# APOE e4 indicators
def apoe_e4_dose(s: str) -> int:
    s = str(s).lower()
    return s.count("e4")

df["APOE_e4_dose"]    = df["APOE"].apply(apoe_e4_dose).astype(int)
df["APOE_e4_carrier"] = (df["APOE_e4_dose"] > 0).astype(int)

# Flag any e2 carriers (for sensitivity analyses)
df["APOE_has_e2"] = df["APOE"].str.contains("e2", case=False, na=False).astype(int)

# Z-transform chosen auditory variables for standardized-effect models
aud_cols = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
for c in aud_cols:
    df[c+"_z"] = (df[c] - df[c].mean())/df[c].std()


### 3) Primary APOE analyses (exclude e2 carriers)


In [15]:
df_no_e2 = df[df['APOE_has_e2'] == 0].copy()
print(f"N with e2 carriers excluded: {df_no_e2.shape[0]}")
print("APOE e4 carrier counts (no e2):")
print(df_no_e2['APOE_e4_carrier'].value_counts())


N with e2 carriers excluded: 178
APOE e4 carrier counts (no e2):
APOE_e4_carrier
0    128
1     50
Name: count, dtype: int64


#### 3A) Adjusted linear models (auditory outcome ~ APOE e4 + age + sex + education ± PAL)


In [16]:
# With cognition (PAL) included as covariate:
covs = "age + C(HighestEducation) + UnencryptedGender + PALTEA28z"
outcomes = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_pal = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
# FDR across the five APOE tests
res_apoe_pal["p_FDR"] = multipletests(res_apoe_pal["p"], method="fdr_bh")[1]
res_apoe_pal


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.549102,1.345960,-2.088931,3.187136,0.683301,0.047593,0.854126
1,slb,-0.956155,0.679206,-2.287374,0.375063,0.159203,0.239765,0.398009
2,awm_freq,-0.044305,0.027777,-0.098747,0.010138,0.110711,0.118912,0.398009
3,awm_am,-0.004070,0.005020,-0.013909,0.005769,0.417514,0.133980,0.695857
4,gmsi_tot,0.075234,3.502567,-6.789671,6.940140,0.982863,0.146105,0.982863


In [17]:
# Without cognition (remove PAL from covariates):
covs = "age + C(HighestEducation) + UnencryptedGender"
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_no_pal = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
res_apoe_no_pal["p_FDR"] = multipletests(res_apoe_no_pal["p"], method="fdr_bh")[1]
res_apoe_no_pal


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.692027,1.366701,-1.986659,3.370713,0.612612,0.030851,0.765765
1,slb,-0.792402,0.697346,-2.159174,0.574371,0.255826,0.179245,0.639566
2,awm_freq,-0.041748,0.027439,-0.095527,0.012032,0.128143,0.111622,0.639566
3,awm_am,-0.003099,0.004932,-0.012766,0.006568,0.529808,0.101104,0.765765
4,gmsi_tot,0.404976,3.417891,-6.293968,7.103920,0.905682,0.134371,0.905682


In [18]:
# Also run with education removed:
covs = "age + UnencryptedGender"
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_no_edu = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
res_apoe_no_edu["p_FDR"] = multipletests(res_apoe_no_edu["p"], method="fdr_bh")[1]
res_apoe_no_edu


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.607992,1.356455,-2.050611,3.266594,0.653994,0.018844,0.817492
1,slb,-0.844956,0.701691,-2.220245,0.530332,0.228523,0.159476,0.571308
2,awm_freq,-0.041479,0.028341,-0.097027,0.014069,0.143314,0.037545,0.571308
3,awm_am,-0.003184,0.004896,-0.012779,0.006412,0.515510,0.057506,0.817492
4,gmsi_tot,0.025453,3.587751,-7.006409,7.057316,0.994339,0.000993,0.994339


#### 3B) Unadjusted group comparisons (carriers vs non-carriers)


In [19]:
def group_stats(series, group):
    a = series[group==0].dropna()
    b = series[group==1].dropna()
    mean_a, sd_a, n_a = a.mean(), a.std(ddof=1), a.size
    mean_b, sd_b, n_b = b.mean(), b.std(ddof=1), b.size
    # Welch t
    t, p = stats.ttest_ind(a, b, equal_var=False)
    # CI for diff using Welch SE
    diff = mean_b - mean_a
    # simple normal-approx CI (ok here): 
    se = np.sqrt(sd_a**2/n_a + sd_b**2/n_b)
    ci_low, ci_high = diff - 1.96*se, diff + 1.96*se
    return mean_a, sd_a, n_a, mean_b, sd_b, n_b, diff, ci_low, ci_high, p

group = df_no_e2["APOE_e4_carrier"]
rows=[]
for y in outcomes:
    rows.append((y, *group_stats(df_no_e2[y], group)))
res_groups = pd.DataFrame(rows, columns=[
    "Outcome","mean_non","sd_non","n_non","mean_car","sd_car","n_car",
    "diff_car_minus_non","CI_low","CI_high","p"
])
res_groups["p_FDR"] = multipletests(res_groups["p"], method="fdr_bh")[1]
res_groups


,Outcome,mean_non,sd_non,n_non,mean_car,sd_car,n_car,diff_car_minus_non,CI_low,CI_high,p,p_FDR
0,dm,2.499213,5.891176,127,3.304000,8.641357,50,0.804787,-1.800418,3.409992,0.546889,0.701970
1,slb,-1.021875,3.505967,128,-1.448000,4.678664,50,-0.426125,-1.858170,1.005920,0.561576,0.701970
2,awm_freq,0.278407,0.176577,128,0.240833,0.164976,50,-0.037574,-0.092591,0.017444,0.183897,0.701970
3,awm_am,0.086324,0.031935,128,0.082652,0.028863,50,-0.003672,-0.013399,0.006056,0.461172,0.701970
4,gmsi_tot,142.531250,20.464656,128,142.604167,22.425991,48,0.072917,-7.194825,7.340658,0.984361,0.984361


### 4) Auditory inter-relationships & link to PAL


#### 4A) Pairwise Pearson correlations among auditory variables


In [20]:
aud = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
rows=[]
for i,a in enumerate(aud):
    for b in aud[i+1:]:
        d = df[[a,b]].dropna()
        r,p = stats.pearsonr(d[a], d[b])
        rows.append([a,b,r,p])
corr_aud = pd.DataFrame(rows, columns=["Var1","Var2","r","p"])
corr_aud


,Var1,Var2,r,p
0,dm,slb,0.325928,0.000002
1,dm,awm_freq,0.268555,0.000121
2,dm,awm_am,0.234195,0.000844
3,dm,gmsi_tot,0.075049,0.293334
4,slb,awm_freq,0.128697,0.068637
5,slb,awm_am,0.221781,0.001555
6,slb,gmsi_tot,0.125381,0.077641
7,awm_freq,awm_am,0.320262,0.000004
8,awm_freq,gmsi_tot,0.271471,0.000105
9,awm_am,gmsi_tot,0.094720,0.183265


#### 4B) Correlations of z-auditory variables with PALTEA28z


In [21]:
rows=[]
for a in aud:
    r,p = stats.pearsonr(df[a+"_z"].dropna(), df.loc[df[a+"_z"].notna(),"PALTEA28z"])
    rows.append([a, r, p])
corr_pal = pd.DataFrame(rows, columns=["Auditory","r_with_PALTEA28z","p"])
corr_pal["p_FDR"] = multipletests(corr_pal["p"], method="fdr_bh")[1]
corr_pal


,Auditory,r_with_PALTEA28z,p,p_FDR
0,dm,0.119691,0.091383,0.152305
1,slb,0.288820,0.000032,0.000160
2,awm_freq,0.088717,0.210421,0.210421
3,awm_am,0.167102,0.017740,0.044350
4,gmsi_tot,0.097581,0.170328,0.210421


### 5) Linear models: PALTEA28z as outcome, one auditory z-predictor at a time


In [22]:
aud_z = [c+"_z" for c in aud]  # dm_z, slb_z, awm_freq_z, awm_am_z, gmsi_tot_z
def run_palteaz_models(data, covariates):
    rows=[]
    for var, label in zip(aud_z, aud):
        formula = f"PALTEA28z ~ {var} + age + UnencryptedGender{covariates}"
        m = smf.ols(formula, data=data).fit(cov_type="HC3")
        beta = m.params[var]; se=m.bse[var]; p=m.pvalues[var]
        (ci_low, ci_high) = m.conf_int().loc[var]
        rows.append([label, beta, se, ci_low, ci_high, p, m.rsquared])
    out = pd.DataFrame(rows, columns=["Auditory","Coef","SE","CI_low","CI_high","p","R2"])
    out["p_FDR"] = multipletests(out["p"], method="fdr_bh")[1]
    return out

# Baseline (with education; no APOE), full data:
res_pal_baseline = run_palteaz_models(df, covariates=" + C(HighestEducation)")
print("--- Baseline PAL Models ---")
print(res_pal_baseline)

# No education:
res_pal_noedu    = run_palteaz_models(df, covariates="")
print("\n--- PAL Models (No Education) ---")
print(res_pal_noedu)

# With APOE:
res_pal_withapoe = run_palteaz_models(df, covariates=" + C(HighestEducation) + APOE_e4_carrier")
print("\n--- PAL Models (With APOE) ---")
print(res_pal_withapoe)

# No education, with APOE:
res_pal_noedu_ap = run_palteaz_models(df, covariates=" + APOE_e4_carrier")
print("\n--- PAL Models (No Education, With APOE) ---")
print(res_pal_noedu_ap)


--- Baseline PAL Models ---
   Auditory      Coef        SE    CI_low   CI_high         p        R2  \
0        dm  0.077932  0.054804 -0.029481  0.185346  0.155019  0.099975   
1       slb  0.209741  0.061676  0.088858  0.330624  0.000672  0.143113   
2  awm_freq  0.059325  0.065334 -0.068726  0.187377  0.363858  0.095931   
3    awm_am  0.142922  0.063648  0.018175  0.267669  0.024735  0.116053   
4  gmsi_tot  0.074204  0.067090 -0.057290  0.205699  0.268710  0.099674   

      p_FDR  
0  0.258366  
1  0.003361  
2  0.363858  
3  0.061837  
4  0.335887  

--- PAL Models (No Education) ---
   Auditory      Coef        SE    CI_low   CI_high         p        R2  \
0        dm  0.078223  0.053722 -0.027069  0.183516  0.145370  0.071560   
1       slb  0.204078  0.067749  0.071292  0.336865  0.002593  0.111155   
2  awm_freq  0.053743  0.060027 -0.063908  0.171394  0.370621  0.065373   
3    awm_am  0.133441  0.060407  0.015045  0.251837  0.027173  0.084722   
4  gmsi_tot  0.075020  0.06

### 6) (Optional) Subjective memory secondary models
If desired, add MemoryProblems (0/1) to the PAL models or run logistic models predicting MemoryProblems from auditory metrics; not part of primary conclusions.


### 7) Reporting checks (sanity)
- Report Ns used per model: `df[model_vars].dropna().shape[0]`
- Echo APOE carrier counts (with and without e2 exclusion).
- Verify coefficient/sign table matches expected ranges above.


### 9) Key conclusions to reproduce

- **APOE analyses (excluding e2):** No significant effect of APOE e4 status on dm, slb, awm_freq, awm_am, or gmsi_tot after adjustment (and likewise unadjusted group tests).
- **Auditory inter-relationships:** Moderate positive associations, notably awm_freq–awm_am (r≈0.32), dm–slb (r≈0.33), dm–awm_freq (r≈0.27).
- **Auditory ↔ PALTEA28z (correlational):** slb_z (r≈0.289, FDR≈1.6e-4) and awm_am_z (r≈0.167, FDR≈0.044) positively related to PALTEA28z; others small/NS.
- **PAL models (adjusted):** slb_z is a robust positive predictor of PALTEA28z after age, sex, education (FDR-significant). awm_am_z is positive but borderline (p≈0.02–0.03; FDR ≈ 0.058–0.068, not passing 0.05).
- **Sensitivity (remove education / add APOE / include e2 carriers):** Estimates barely change; conclusions unchanged.
